# Cerebellar Disorders of Eye Movement

The cerebellum has distinct regional specialisations with different oculomotor roles.

## Anatomy overview

```
Flocculus / Paraflocculus
  → clamps Neural Integrator (NPH)      → GEN when damaged
  → drives NI null-point adaptation     → Rebound nystagmus when damaged
  → encodes retinal velocity for pursuit → Impaired smooth pursuit when damaged

Nodulus / Uvula  (vestibulocerebellum)
  → gates Velocity Storage adaptation   → prolonged OKAN, PAN when damaged
  → modulates otolith-canal integration → tilt/translation confusion
```

## State architecture

```
Neural Integrator (NI):  [x_L(3) | x_R(3) | x_null(3)]   — NPH left/right pops + null
Velocity Storage  (VS):  [x_L(3) | x_R(3) | x_null(3)]   — VN  left/right pops + null

NI net:  x_L − x_R  (eye position command, deg)
VS net:  x_L − x_R  (head velocity estimate, deg/s)

NI dynamics:  d(x_net)/dt = −(1/τ_i) · x_net + u_vel          ← flocculus sets τ_i
NI null:      dx_null/dt  =  (x_net − x_null) / τ_ni_adapt    ← flocculus drives adaptation
VS dynamics:  d(x_net)/dt = −(1/τ_vs) · (x_net − x_null) + B @ u_in
VS null:      dx_null/dt  =  (w_est  − x_null) / τ_vs_adapt   ← nodulus gates τ_vs_adapt
```

## Literature

| Claim | Source | Confidence |
|-------|--------|------------|
| Flocculus clamps NI → GEN after lesion | Cannon & Robinson (1985) *Biol Cybern* | ✅ |
| Rebound nystagmus — cerebellar sign | Zee et al. (1980) *Brain* | ✅ |
| Flocculus drives smooth pursuit | Stone & Lisberger (1990) *J Neurophysiol* | ✅ |
| Nodulus ablation → prolonged OKAN, PAN-like | Cohen et al. (1992) *J Neurophysiol* | ✅ |
| PAN period ~2 min in humans | Leigh et al. (1981) *Neurology* | ✅ |

## Sections

| Section | Structure | Conditions |
|---------|-----------|------------|
| 1 | Flocculus/Paraflocculus | GEN, rebound, impaired pursuit (combined) |
| 2 | Flocculus + UVH | Bruns nystagmus |
| 3 | Nodulus/Uvula | Prolonged OKAN, VS null adaptation |
| 4 | Nodulus (future) | Periodic Alternating Nystagmus (PAN) |

In [ ]:
import numpy as np
import jax.numpy as jnp
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

from oculomotor.sim.simulator import (
    PARAMS_DEFAULT, with_brain, with_sensory, simulate, SimConfig,
    _IDX_VS_L, _IDX_VS_R, _IDX_VS_NULL,
    _IDX_NI_L, _IDX_NI_R, _IDX_NI_NULL,
)
from oculomotor.sim import kinematics as km
from oculomotor import __version__
from oculomotor.analysis import (
    vs_net as _vs_net3, ni_net as _ni_net3,
    vs_null as _vs_null3, ni_null as _ni_null3,
    extract_burst as _extract_burst3,
    extract_spv, ax_fmt,
)

print(f'oculomotor {__version__}')
THETA = PARAMS_DEFAULT
print(f'τ_i={THETA.brain.tau_i}s  τ_ni_adapt={THETA.brain.tau_ni_adapt}s')
print(f'τ_vs={THETA.brain.tau_vs}s  τ_vs_adapt={THETA.brain.tau_vs_adapt}s')

%matplotlib inline
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size']  = 9

SR  = 200.0
DT  = 1.0 / SR

# Yaw-only scalar wrappers (full 3D via _vs_net3, _ni_net3, etc.)
def ni_net(states):   return _ni_net3(states)[:, 0]
def ni_null(states):  return _ni_null3(states)[:, 0]
def vs_net(states):   return _vs_net3(states)[:, 0]
def vs_null(states):  return _vs_null3(states)[:, 0]
def extract_burst(states, theta): return _extract_burst3(states, theta)[:, 0]
def eye_pos(states):  return np.array(states.plant[:, 0])
def eye_vel(states):  return np.gradient(eye_pos(states), DT)

def slow_phase(t, ev, burst, threshold=10.0, margin_s=0.05):
    return extract_spv(t, ev, burst, burst_threshold=threshold, margin_s=margin_s)

---
## 1. Flocculus / Paraflocculus — Gaze-Evoked Nystagmus, Rebound & Pursuit

The flocculus and paraflocculus (FL/PFL) modulate three oculomotor circuits:

| Circuit | Healthy | After FL/PFL lesion |
|---------|---------|---------------------|
| Neural Integrator clamping (NPH) | τ_i ≈ 25 s | τ_i ↓ → **gaze-evoked nystagmus** (GEN) |
| NI null-point adaptation | x_null tracks NI over ~20 s | Null persists after gaze shift → **rebound nystagmus** |
| Smooth pursuit drive | gain ≈ 1 at 15 deg/s | K_pursuit ↓ → cogwheel pursuit, catch-up saccades |

**Note:** GEN and rebound coexist in the same patient — a single protocol with hold + return reveals both.

**Simulation parameters:**
- Healthy: τ_i = 25 s, τ_ni_adapt = 20 s, K_pursuit = 4, K_phasic = 5
- FL/PFL lesion: τ_i = 2 s, K_pursuit = 0.5, K_phasic = 1

**Refs:** Zee et al. (1980) *Brain* ✅; Cannon & Robinson (1985) *Biol Cybern* ✅; Stone & Lisberger (1990) ✅

In [ ]:
%%time
# === Flocculus/paraflocculus lesion parameters ===
THETA_FL = with_brain(THETA,
    tau_i=2.0,              # leaky neural integrator → GEN + rebound
    K_pursuit=0.5,          # reduced pursuit gain
    K_phasic_pursuit=1.0,   # reduced phasic pursuit onset
)

# ── Protocol 1: centre (1 s) → 20° hold (20 s) → centre (15 s) ──────────────
# t[0] = 0 → target at centre → warmup (default 3 s) initialises at centre
PRE_DUR  = 1.0; HOLD_DUR = 20.0; POST_DUR = 15.0
TOTAL_FL = PRE_DUR + HOLD_DUR + POST_DUR

t_fl    = jnp.arange(0.0, TOTAL_FL, DT)
T_fl    = len(t_fl)
t_fl_np = np.array(t_fl)

tx_fl  = jnp.where(t_fl < PRE_DUR, 0.0,
         jnp.where(t_fl < PRE_DUR + HOLD_DUR, jnp.tan(jnp.radians(20.0)), 0.0))
pt3_fl = jnp.stack([tx_fl, jnp.zeros(T_fl), jnp.ones(T_fl)], axis=1).astype(jnp.float32)
lv_fl  = jnp.zeros((T_fl, 3), dtype=jnp.float32)
tgt_fl = km.build_target(t_fl, lin_pos=pt3_fl, lin_vel=lv_fl)

max_fl = int(TOTAL_FL / 0.001) + 1000
st_fl_h = simulate(THETA,    t_fl, target=tgt_fl, scene_present_array=jnp.ones(T_fl),
                   max_steps=max_fl, return_states=True)
st_fl_c = simulate(THETA_FL, t_fl, target=tgt_fl, scene_present_array=jnp.ones(T_fl),
                   max_steps=max_fl, return_states=True)

burst_fl_h = extract_burst(st_fl_h, THETA)
burst_fl_c = extract_burst(st_fl_c, THETA_FL)
spv_fl_h   = slow_phase(t_fl_np, eye_vel(st_fl_h), burst_fl_h)
spv_fl_c   = slow_phase(t_fl_np, eye_vel(st_fl_c), burst_fl_c)
t_hold_end = PRE_DUR + HOLD_DUR

# ── Protocol 2: smooth pursuit ramp ──────────────────────────────────────────
RAMP_VEL = 15.0; RAMP_DUR = 4.0; PUR_TOTAL = 7.0

t_pur    = jnp.arange(0.0, PUR_TOTAL, DT)
T_pur    = len(t_pur)
t_pur_np = np.array(t_pur)

angle_pur = jnp.clip(RAMP_VEL * t_pur, 0.0, RAMP_VEL * RAMP_DUR)
tx_pur  = jnp.tan(jnp.radians(angle_pur))
vx_pur  = jnp.where(t_pur < RAMP_DUR, RAMP_VEL * jnp.pi / 180.0, 0.0)  # m/s at depth=1 m
pt3_pur = jnp.stack([tx_pur, jnp.zeros(T_pur), jnp.ones(T_pur)], axis=1).astype(jnp.float32)
lv_pur  = jnp.stack([vx_pur, jnp.zeros(T_pur), jnp.zeros(T_pur)], axis=1).astype(jnp.float32)
tgt_pur = km.build_target(t_pur, lin_pos=pt3_pur, lin_vel=lv_pur)

max_pur = int(PUR_TOTAL / 0.001) + 500
st_pur_h = simulate(THETA,    t_pur, target=tgt_pur, scene_present_array=jnp.ones(T_pur),
                    max_steps=max_pur, return_states=True)
st_pur_c = simulate(THETA_FL, t_pur, target=tgt_pur, scene_present_array=jnp.ones(T_pur),
                    max_steps=max_pur, return_states=True)

print('Done.')

In [ ]:
C_h = '#2166ac'; C_c = '#d6604d'

fig = plt.figure(figsize=(16, 12))
fig.suptitle(
    'Flocculus / Paraflocculus Lesion — Gaze-Evoked Nystagmus, Rebound & Pursuit\n'
    f'(τ_i: {THETA.brain.tau_i:.0f}→2 s  |  K_pursuit: {THETA.brain.K_pursuit:.1f}→0.5)',
    fontsize=11, fontweight='bold')

gs = GridSpec(3, 2, figure=fig, hspace=0.52, wspace=0.38)

# ── Left column: GEN + Rebound ────────────────────────────────────────────────
ax_ep  = fig.add_subplot(gs[0, 0])
ax_spv = fig.add_subplot(gs[1, 0], sharex=ax_ep)
ax_ni  = fig.add_subplot(gs[2, 0], sharex=ax_ep)

ax_ep.plot(t_fl_np, eye_pos(st_fl_h), color=C_h, lw=1.5, label='Healthy')
ax_ep.plot(t_fl_np, eye_pos(st_fl_c), color=C_c, lw=1.5, label='FL/PFL lesion')
ax_ep.axvline(PRE_DUR,    color='gray', lw=0.6, ls=':', alpha=0.5)
ax_ep.axvline(t_hold_end, color='k',   lw=0.7, ls='--', alpha=0.4)
ax_ep.axhline(20, color='gray', lw=0.7, ls=':', alpha=0.4)
ax_fmt(ax_ep, ylabel='Eye position (deg)')
ax_ep.legend(fontsize=7, loc='upper right')
ax_ep.set_title('Gaze-evoked nystagmus + rebound — eye position', fontsize=9)

ax_spv.plot(t_fl_np, spv_fl_h, color=C_h, lw=2.0, label='Healthy SPV')
ax_spv.plot(t_fl_np, spv_fl_c, color=C_c, lw=2.0, label='FL/PFL lesion SPV')
ax_spv.axvline(PRE_DUR,    color='gray', lw=0.6, ls=':', alpha=0.5)
ax_spv.axvline(t_hold_end, color='k',   lw=0.7, ls='--', alpha=0.4)
ax_fmt(ax_spv, ylabel='SPV (deg/s)', ylim=(-20, 20))
ax_spv.legend(fontsize=7)
ax_spv.set_title('Slow-phase velocity (±20 deg/s) — GEN centripetal → rebound eccentric', fontsize=9)

ax_ni.plot(t_fl_np, ni_net(st_fl_c),  color=C_c,      lw=1.5, label='Neural Integrator net (lesion)')
ax_ni.plot(t_fl_np, ni_null(st_fl_c), color='#e08214', lw=1.5, label='Neural Integrator null')
ax_ni.plot(t_fl_np, ni_net(st_fl_h),  color=C_h,      lw=0.8, ls='--', alpha=0.4, label='Healthy (ref)')
ax_ni.axvline(PRE_DUR,    color='gray', lw=0.6, ls=':', alpha=0.5)
ax_ni.axvline(t_hold_end, color='k',   lw=0.7, ls='--', alpha=0.4)
ax_fmt(ax_ni, ylabel='Neural Integrator state (deg)', xlabel='Time (s)')
ax_ni.legend(fontsize=7)
ax_ni.set_title('Neural Integrator state — null drifts during hold; persists after return', fontsize=9)

# ── Right column: Smooth Pursuit ──────────────────────────────────────────────
ax_pp  = fig.add_subplot(gs[0, 1])
ax_pv  = fig.add_subplot(gs[1, 1], sharex=ax_pp)
ax_txt = fig.add_subplot(gs[2, 1])

tgt_deg = np.degrees(np.arctan(np.array(tx_pur)))
tgt_vel = np.where(t_pur_np < RAMP_DUR, RAMP_VEL, 0.0)

ax_pp.plot(t_pur_np, tgt_deg,           color='gray', lw=1.2, ls='--', label='Target')
ax_pp.plot(t_pur_np, eye_pos(st_pur_h), color=C_h,   lw=1.5, label='Healthy')
ax_pp.plot(t_pur_np, eye_pos(st_pur_c), color=C_c,   lw=1.5, label='FL/PFL lesion')
ax_pp.axvline(RAMP_DUR, color='k', lw=0.7, ls='--', alpha=0.4)
ax_fmt(ax_pp, ylabel='Eye / target position (deg)')
ax_pp.legend(fontsize=7)
ax_pp.set_title(f'Smooth pursuit — {RAMP_VEL:.0f} deg/s ramp', fontsize=9)

ax_pv.plot(t_pur_np, tgt_vel,           color='gray', lw=1.2, ls='--', label='Target velocity')
ax_pv.plot(t_pur_np, eye_vel(st_pur_h), color=C_h,   lw=1.5, label='Healthy')
ax_pv.plot(t_pur_np, eye_vel(st_pur_c), color=C_c,   lw=1.5, label='FL/PFL lesion')
ax_pv.axvline(RAMP_DUR, color='k', lw=0.7, ls='--', alpha=0.4)
ax_fmt(ax_pv, ylabel='Eye / target velocity (deg/s)', xlabel='Time (s)')
ax_pv.legend(fontsize=7)
ax_pv.set_title('Pursuit velocity — reduced gain, catch-up saccades in lesion', fontsize=9)

ax_txt.axis('off')
ax_txt.text(0.05, 0.95,
    'FL/PFL lesion parameters\n\n'
    'Neural Integrator leak:\n'
    '  τ_i : 25 s → 2 s\n\n'
    'NI null adaptation:\n'
    '  τ_ni_adapt : 20 s (unchanged)\n\n'
    'Smooth pursuit:\n'
    '  K_pursuit  : 4.0 → 0.5\n'
    '  K_phasic   : 5.0 → 1.0\n\n'
    'Refs:\n'
    '  Zee et al. (1980) Brain\n'
    '  Cannon & Robinson (1985)\n'
    '  Stone & Lisberger (1990)',
    transform=ax_txt.transAxes, fontsize=9, va='top', family='monospace',
    bbox=dict(boxstyle='round,pad=0.5', facecolor='#f5f5f5', alpha=0.8))

plt.show()

---
## 2. Bruns Nystagmus — Flocculus Lesion + Unilateral Hypofunction

Bruns nystagmus is seen with cerebellopontine angle tumours (acoustic neuroma, meningioma):
- **Ipsilesional gaze:** large-amplitude, slow-frequency nystagmus — GEN (leaky Neural Integrator) amplified by peripheral VS imbalance
- **Contraversive gaze:** small-amplitude, high-frequency nystagmus — pure rebound nystagmus

Model: leaky Neural Integrator + null adaptation + unilateral left VN hypofunction (UVH).

**Parameters:** `τ_i = 3.0 s`, `τ_ni_adapt = 15.0 s`, left UVH (canal gain × 0.1)

**Expected:**
- Spontaneous rightward nystagmus at rest (from UVH)
- Large, slow nystagmus during leftward (ipsilesional) gaze
- Smaller, faster nystagmus + rebound during rightward (contraversive) gaze

**Ref:** Bruns (1902); Leigh & Zee (2015) ✅

In [ ]:
%%time
from oculomotor.sim.simulator import with_uvh

# Bruns: leaky Neural Integrator + active null adaptation + left UVH
THETA_BRUNS = with_uvh(
    with_brain(THETA, tau_i=3.0, tau_ni_adapt=15.0),
    side='left', canal_gain_frac=0.1
)

SEG = 12.0
T_TOTAL = 4 * SEG
t_br    = jnp.arange(0.0, T_TOTAL, DT)
T_br    = len(t_br)
t_br_np = np.array(t_br)

def _target_steps(t_arr, segs):
    tx = jnp.zeros_like(t_arr)
    for t_start, deg in segs:
        tx = jnp.where(t_arr >= t_start, jnp.tan(jnp.radians(deg)), tx)
    return tx

tx_br  = _target_steps(t_br, [(0, 0), (SEG, 20), (2*SEG, 0), (3*SEG, -20)])
pt3_br = jnp.stack([tx_br, jnp.zeros(T_br), jnp.ones(T_br)], axis=1).astype(jnp.float32)
lv_br  = jnp.zeros((T_br, 3), dtype=jnp.float32)
tgt_br = km.build_target(t_br, lin_pos=pt3_br, lin_vel=lv_br)

max_br = int(T_TOTAL / 0.001) + 2000
st_br  = simulate(THETA_BRUNS, t_br, target=tgt_br,
                  scene_present_array=jnp.ones(T_br),
                  max_steps=max_br, return_states=True)

burst_br = extract_burst(st_br, THETA_BRUNS)
spv_br   = slow_phase(t_br_np, eye_vel(st_br), burst_br)
print('Done.')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
fig.suptitle('Bruns Nystagmus — Leaky Neural Integrator + Rebound + Left UVH',
             fontsize=10, fontweight='bold')

C_br = '#35978f'

axes[0].plot(t_br_np, eye_pos(st_br), color=C_br, lw=1.2, label='Eye pos')
target_deg = np.degrees(np.arctan(np.array(tx_br)))
axes[0].plot(t_br_np, target_deg, color='gray', lw=0.8, ls='--', label='Target')
for seg in range(1, 4):
    axes[0].axvline(seg*SEG, color='k', lw=0.5, ls='--', alpha=0.3)
ax_fmt(axes[0], ylabel='Eye position (deg)')
axes[0].legend(fontsize=7)
axes[0].set_title('Eye position — large drift during ipsilesional (leftward) gaze')

axes[1].plot(t_br_np, spv_br, color=C_br, lw=2.0)
for seg in range(1, 4):
    axes[1].axvline(seg*SEG, color='k', lw=0.5, ls='--', alpha=0.3)
ax_fmt(axes[1], ylabel='SPV (deg/s)', ylim=(-60, 60))
axes[1].set_title('SPV — spontaneous at rest; large ipsilesional; rebound on return to centre')

axes[2].plot(t_br_np, ni_net(st_br),  color=C_br,     lw=1.5, label='Neural Integrator net')
axes[2].plot(t_br_np, ni_null(st_br), color='#e08214', lw=1.5, ls='--', label='Neural Integrator null')
for seg in range(1, 4):
    axes[2].axvline(seg*SEG, color='k', lw=0.5, ls='--', alpha=0.3)
ax_fmt(axes[2], ylabel='Neural Integrator state (deg)', xlabel='Time (s)')
axes[2].legend(fontsize=7)
axes[2].set_title('Neural Integrator state — null follows eye position; persists after each shift → rebound')

labels = ['centre', '+20° contra', 'centre', '−20° ipsi']
for i, lbl in enumerate(labels):
    axes[0].text(i*SEG + SEG/2, -30, lbl, ha='center', fontsize=7, color='gray')

fig.tight_layout()
plt.show()

---
## 3. Nodulus / Uvula — Velocity Storage Null Adaptation and OKAN

The **nodulus and uvula** (vestibulocerebellum) regulate the adaptation time constant of velocity storage (`τ_vs_adapt`).

- **Intact nodulus:** `τ_vs_adapt` is long (≥600 s) → null barely builds during normal activity
- **Nodulus lesion:** `τ_vs_adapt` shortens → null builds during sustained OKN/VOR → **prolonged OKAN**, possible PAN

**Mechanism:**
During sustained OKN (30 deg/s), the VS null builds toward `w_est`.  
After lights off, VS decays toward null (not 0) → OKAN outlasts the VS time constant alone.  
With very short `τ_vs_adapt`, the null can overshoot → after-rebound direction reversal.

**Parameters:**
- Healthy: `τ_vs_adapt = 600 s`
- Nodulus lesion: `τ_vs_adapt = 60 s`

**Ref:** Cohen et al. (1992) *J Neurophysiol* (nodulus ablation) ✅

In [ ]:
%%time
THETA_NOD = with_brain(THETA, tau_vs_adapt=60.0)  # nodulus lesion: faster VS null adaptation

ON_DUR = 30.0; OFF_DUR = 60.0
t_vn    = jnp.arange(0.0, ON_DUR + OFF_DUR, DT)
T_vn    = len(t_vn)
t_vn_np = np.array(t_vn)

v_sc_3d   = jnp.zeros((T_vn, 3)).at[:, 0].set(jnp.where(t_vn < ON_DUR, 30.0, 0.0))
sc_pr     = jnp.where(t_vn < ON_DUR, 1.0, 0.0)
scene_okn = km.build_kinematics(t_vn, rot_vel=v_sc_3d)

max_vn = int((ON_DUR + OFF_DUR) / 0.001) + 1000
st_vn_h = simulate(THETA,     t_vn,
                   scene=scene_okn, scene_present_array=sc_pr,
                   target_present_array=jnp.zeros(T_vn),
                   max_steps=max_vn, return_states=True)
st_vn_a = simulate(THETA_NOD, t_vn,
                   scene=scene_okn, scene_present_array=sc_pr,
                   target_present_array=jnp.zeros(T_vn),
                   max_steps=max_vn, return_states=True)

burst_vn_h = extract_burst(st_vn_h, THETA)
burst_vn_a = extract_burst(st_vn_a, THETA_NOD)
spv_vn_h   = slow_phase(t_vn_np, eye_vel(st_vn_h), burst_vn_h)
spv_vn_a   = slow_phase(t_vn_np, eye_vel(st_vn_a), burst_vn_a)
print('Done.')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
fig.suptitle(
    f'Nodulus/Uvula Lesion — Prolonged OKAN via Velocity Storage Null Adaptation\n'
    f'(τ_vs_adapt: {THETA.brain.tau_vs_adapt:.0f} s  →  {THETA_NOD.brain.tau_vs_adapt:.0f} s)',
    fontsize=10, fontweight='bold')

C_h = '#2166ac'; C_a = '#d6604d'

axes[0].plot(t_vn_np, spv_vn_h, color=C_h, lw=2.0,
             label=f'Healthy  τ_vs_adapt = {THETA.brain.tau_vs_adapt:.0f} s')
axes[0].plot(t_vn_np, spv_vn_a, color=C_a, lw=2.0,
             label=f'Nodulus lesion  τ_vs_adapt = {THETA_NOD.brain.tau_vs_adapt:.0f} s')
axes[0].axvline(ON_DUR, color='k', lw=0.7, ls='--', alpha=0.5, label='Lights off')
axes[0].axhline(30, color='gray', lw=0.7, ls=':', alpha=0.4)
ax_fmt(axes[0], ylabel='SPV (deg/s)')
axes[0].legend(fontsize=7)
axes[0].set_title('OKN + OKAN — nodulus lesion prolongs OKAN after lights off')

axes[1].plot(t_vn_np, -vs_net(st_vn_h), color=C_h, lw=1.5, label='Velocity Storage net (healthy)')
axes[1].plot(t_vn_np, -vs_net(st_vn_a), color=C_a, lw=1.5, label='Velocity Storage net (nodulus lesion)')
axes[1].axvline(ON_DUR, color='k', lw=0.7, ls='--', alpha=0.5)
ax_fmt(axes[1], ylabel='Velocity Storage net (deg/s)')
axes[1].legend(fontsize=7)
axes[1].set_title('Velocity Storage state — same charging; decays toward null (not 0) → extended OKAN')

axes[2].plot(t_vn_np, -vs_null(st_vn_h), color=C_h, lw=1.0, ls='--', alpha=0.5,
             label='VS null (healthy, negligible)')
axes[2].plot(t_vn_np, -vs_null(st_vn_a), color=C_a, lw=2.0, label='VS null (nodulus lesion)')
axes[2].axvline(ON_DUR, color='k', lw=0.7, ls='--', alpha=0.5)
ax_fmt(axes[2], ylabel='Velocity Storage null (deg/s)', xlabel='Time (s)')
axes[2].legend(fontsize=7)
axes[2].set_title('VS null — builds during OKN; residual shifts decay target → longer OKAN')

fig.tight_layout()
plt.show()

---
## 4. Future: Periodic Alternating Nystagmus (PAN)

**Mechanism (proposed):**
Nodulus/uvula adapts to ongoing spontaneous nystagmus, attempting to cancel it.  
With INHIBITORY VS null coupling (null opposes state rather than sustaining it),  
the null overshoots → nystagmus reverses → null adapts again → ~2-min oscillation.

**Current model limitation:**
Null coupling here is EXCITATORY: `b_eff = b ± x_null/2` → null sustains state → OKAN extension.  
PAN requires INHIBITORY coupling: `b_eff = b ∓ x_null/2` → null opposes state → oscillation.

**Implementation plan:**
- Add `sign_vs_null: float = +1.0` to BrainParams (default = excitatory / OKAN extension)
- Set `sign_vs_null = -1.0` for PAN modelling (inhibitory / oscillation)
- Seed with spontaneous nystagmus (UVH) + tune `τ_vs_adapt` (~30–60 s)

**Ref:** Cohen et al. (1988, 1992); Leigh et al. (1981) *Neurology* ✅

In [ ]:
print('PAN simulation pending sign_vs_null parameter — see section description above.')